In [ ]:
!pip install jsonlines
!pip install torch_geometric
!pip install torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━ 144.2/207.5 MB 87.7 MB/s eta 0:00:01

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
DATA_FOLDER = Path('/content/drive/MyDrive/DSCI599/data')

# load data



In [ ]:
csv_files = list(DATA_FOLDER.glob("*.csv"))
for f in csv_files:
  print(f)

In [ ]:
import pandas as pd
df = pd.read_csv(csv_files[0])
df.head()

In [ ]:
print("Number of unique source_ips")
len(df[['int_source_ip']].value_counts())

In [ ]:
print("Number of unique (source_ip, destination_port):")
len(df[['int_source_ip', "destination_port"]].value_counts().index)

In [ ]:
df.groupby(['int_source_ip', "destination_port"])['protocol'].count().mean()

In [ ]:
random_ip = df['int_source_ip'].sample(1).tolist()
df[df['int_source_ip'].isin(random_ip)]

In [ ]:
random_ip = df['int_source_ip'].sample(1).tolist()
df[df['int_source_ip'].isin(random_ip)]

# Load data as a Heterogenous graph

Since lots of the GCN are built for undirected graphs, let's load the data as a heterogenous graph

Plan:
- Load each source ip -> destination port IF they actually have connection in dataset
- Set everything as source ip feature except int_source_ip and destination
- OHE the destination port to set it as categorical
- aggregate the edges, so each source_ip -> destination only have ONE edge
- edge feature is protocol

For each int_source_ip -> (protocol) -> destination_port, get the mean of everything

This to ensure only one edge per source and destination to avoid complications

In [ ]:
grouped_df = df.groupby(['int_source_ip', 'destination_port', 'protocol'])
grouped_df = grouped_df.mean().reset_index(drop=False)
grouped_df.head()

In [ ]:
grouped_df.shape

(7598, 15)

In [ ]:
grouped_df['compromised'].value_counts()

,count
compromised,
0.0,5043
1.0,2555


Get source feature names

In [ ]:
src_node_featnames = df.drop(columns=['int_source_ip', 'destination_port', 'protocol', 'compromised']).columns.tolist()
src_node_featnames

['num_scanned_destination_ips',
 'num_unique_flows',
 'num_packets',
 'same_packet_size_flag',
 'avg_packet_size',
 'num_unique_Bs_scanned',
 'num_unique_Cs_scanned',
 'num_unique_Ds_scanned',
 'num_scanned_24_blocks',
 'num_non_conficker_destinations',
 'time_activity']

Get unique destination ports

In [ ]:
destination_ports = grouped_df['destination_port'].unique()
dest_features = [[port] for port in destination_ports]
dest_port_to_idx = {port: idx for idx, port in enumerate(destination_ports)}

dest_features[:3]

[[23], [53413], [22]]

Get connections

In [ ]:
src_indices = grouped_df['int_source_ip'].index.tolist()
dest_indices = grouped_df['destination_port'].map(lambda port: dest_port_to_idx[port]).tolist()
edge_protocols = grouped_df['protocol'].tolist()

Keep only one edge per source_ip to destination_port

In [ ]:
from torch_geometric.data import HeteroData
import torch

data = HeteroData()
source_ip_feats = grouped_df[src_node_featnames].values
labels = grouped_df['compromised'].values

# assign source edges x and y
data['source_ip'].x = torch.tensor(source_ip_feats, dtype=torch.float)
data['source_ip'].y = torch.tensor(labels, dtype=torch.long)
data['source_ip'].num_nodes = len(source_ip_feats)

# create destination ports
data['dest_ports'].x = torch.tensor(dest_features, dtype=torch.long)
data['dest_ports'].num_nodes = len(dest_features)

# create connections
data['source_ip', 'scans', 'dest_ports'].edge_index = torch.tensor(
    [src_indices, dest_indices], dtype=torch.long
)
data['source_ip', 'scans', 'dest_ports'].edge_attr = torch.tensor(
    edge_protocols, dtype=torch.float
)
data['source_ip', 'scans', 'dest_ports'].edge_weight = torch.tensor(
    torch.ones((len(edge_protocols))), dtype=torch.float
)

data

<ipython-input-136-b893bfefb4a3>:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data['source_ip', 'scans', 'dest_ports'].edge_weight = torch.tensor(


HeteroData(
  source_ip={
    x=[7598, 11],
    y=[7598],
    num_nodes=7598,
  },
  dest_ports={
    x=[599, 1],
    num_nodes=599,
  },
  (source_ip, scans, dest_ports)={
    edge_index=[2, 7598],
    edge_attr=[7598],
    edge_weight=[7598],
  }
)

# Split train test

> https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.transforms.RandomLinkSplit.html#torch_geometric.transforms.RandomLinkSplit

In [ ]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(is_undirected=False,
                            num_val=0.1,
                            num_test=0.2,
                            edge_types=data.edge_types)

# doesn't work!
# train_data, val_data, test_data = transform(data)

# just use pytorch functions
split_perc = 0.8
num_nodes = data['source_ip'].num_nodes
perm = torch.randperm(num_nodes)
train_size = int(split_perc * num_nodes)
train_idx = perm[:train_size]
test_idx = perm[train_size:]

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True

data['source_ip'].train_mask = train_mask
data['source_ip'].test_mask = test_mask

In [ ]:
data

HeteroData(
  source_ip={
    x=[7598, 11],
    y=[7598],
    num_nodes=7598,
  },
  dest_ports={
    x=[599, 1],
    num_nodes=599,
  },
  (source_ip, scans, dest_ports)={
    edge_index=[2, 7598],
    edge_attr=[7598],
  }
)

# Convert to undirected

Most of the GCN works for undirected graphs, so have to make it undirected

In [ ]:
from torch_geometric.transforms import ToUndirected



# Model

> https://pytorch-geometric.readthedocs.io/en/latest/tutorial/heterogeneous.html

In [ ]:
from torch_geometric.nn import SAGEConv, to_hetero
import torch.nn.functional as F

# from link above
class GNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

In [ ]:
num_classes = data['source_ip'].y.max().item() + 1
num_classes

2

In [ ]:
model = GNN(hidden_channels = 64, out_channels=num_classes)
model = to_hetero(model, data.metadata(), aggr='sum')

/usr/local/lib/python3.11/dist-packages/torch_geometric/nn/to_hetero_transformer.py:156: UserWarning: There exist node types ({'source_ip'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  warnings.warn(


ValueError: Cannot generate a graph node 'relu' for type 'source_ip' since it does not exist. Please make sure that all node types get updated during message passing.